# 0. Introduction

> <center> L’obiettivo di questa analisi è verificare se il sentiment giornaliero su Amazon (AMZN), estratto da Reddit e dal database GDELT, contenga informazioni utili per prevedere i rendimenti futuri del titolo.

---

---

# 1.A. Reddit request API: AMZN

In [10]:
import praw
import pandas as pd
from datetime import datetime, timedelta

reddit = praw.Reddit(
    client_id="-",
    client_secret="-",
    user_agent="market-sentiment-research by amicopaolo",
    check_for_async=False
)

subreddits = [
    "stocks",
    "StockMarket",
    "investing",
    "wallstreetbets",
]

query = '"AMZN" OR "Amazon stock" OR "Amazon.com" OR "Amazon shares"'

oggi = datetime.utcnow()
start_dt = oggi - timedelta(days=180)
start_ts = start_dt.timestamp()

rows = []

for name in subreddits:
    sub = reddit.subreddit(name)
    for submission in sub.search(query, sort="new", limit=None):
        created = submission.created_utc
        if created >= start_ts:
            rows.append({
                "subreddit": name,
                "id": submission.id,
                "created_utc": created,
                "created_dt": datetime.utcfromtimestamp(created).isoformat(),
                "title": submission.title,
                "selftext": submission.selftext,
                "score": submission.score,
                "num_comments": submission.num_comments,
                "url": submission.url,
            })
        else:
            break

df = pd.DataFrame(rows)
df.to_csv("reddit_amzn_posts_180d.csv", index=False, encoding="utf-8")
print("Post raccolti:", df.shape[0])

Post raccolti: 279


> <center> La raccolta dei dati Reddit avviene interrogando subreddit a tema finanziario (r/stocks, r/StockMarket, r/investing, r/wallstreetbets) con una query che combina “AMZN” e i principali alias di Amazon.

---

# 2.A. Dataset cleaning

In [11]:
import pandas as pd
from datetime import datetime, timezone

df = pd.read_csv("reddit_amzn_posts_180d.csv")

df["datetime_utc"] = pd.to_datetime(df["created_utc"], unit="s", utc=True)

df["text"] = df["title"].fillna("") + " " + df["selftext"].fillna("")
df["text"] = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()

df["len_text"] = df["text"].str.len()
df = df[df["len_text"] >= 50]

df = df[df["score"] >= 1]
df = df[df["num_comments"] >= 1]

spam_patterns = [
    "whatsapp",
    "telegram",
    "signals",
    "forex broker",
    "join my group",
    "discord.gg",
    "pump and dump",
    "signal group",
]
mask_spam = df["text"].str.lower().str.contains("|".join(spam_patterns))
df = df[~mask_spam]

df = df[df["text"].str.strip() != ""]

df_clean = df[["subreddit", "datetime_utc", "text", "score", "num_comments"]].copy()

df_clean.to_csv("reddit_amzn_posts_180d_clean.csv", index=False, encoding="utf-8")
print("Post totali dopo la pulizia:", df_clean.shape[0])


Post totali dopo la pulizia: 203


> <center> Il dataset grezzo viene filtrato applicando soglie minime su lunghezza del testo, score e numero di commenti, oltre a una blacklist di pattern tipici dello spam (gruppi Telegram/Discord, segnali di trading, ecc.).

# 3.A. Daily sentiment

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

df = pd.read_csv("reddit_amzn_posts_180d_clean.csv")

df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True)
df["date"] = df["datetime_utc"].dt.date

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

finbert = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True,
    truncation=True,
    max_length=512,
)

def score_to_polarity(scores):
    if isinstance(scores, dict):
        scores = [scores]

    if isinstance(scores, list) and len(scores) > 0 and isinstance(scores[0], list):
        scores = scores[0]

    if not (isinstance(scores, list) and len(scores) > 0 and isinstance(scores[0], dict)):
        return 0.0

    d = {s["label"].lower(): s["score"] for s in scores}
    pos = d.get("positive", 0.0)
    neg = d.get("negative", 0.0)
    return pos - neg

df_sample = df.copy()

sentiments = []
for text in df_sample["text"]:
    if not isinstance(text, str) or text.strip() == "":
        sentiments.append(0.0)
        continue
    out = finbert(text)
    sentiments.append(score_to_polarity(out))

df_sample["sentiment"] = sentiments

daily = (
    df_sample.groupby("date")["sentiment"]
    .mean()
    .reset_index()
    .sort_values("date")
)

daily.to_csv("daily_amzn_sentiment_a.csv", index=False, encoding="utf-8")

print("Numero di post analizzati:", len(df_sample))
print("Numero di giorni nella serie:", len(daily))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Numero di post analizzati: 203
Numero di giorni nella serie: 117


> <center> Per ogni post pulito viene applicato FinBERT in modalità “text‑classification” e convertità le probabilità Positive/Negative in uno score di polarità compreso indicativamente tra −1 e +1. <br> Questa serie verrà usata in tutte le analisi successive accanto ai rendimenti di prezzo.

---

---

# 1.B. GDELT Event Database 2.0: AMZN

In [14]:
import requests
import zipfile
import io
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import clear_output

def download_gdelt_events_day(day_str: str) -> pd.DataFrame:
    url = f"http://data.gdeltproject.org/events/{day_str}.export.CSV.zip"
    clear_output(wait=True)
    print(f"Scarico {url} ...")

    r = requests.get(url)
    r.raise_for_status()

    z = zipfile.ZipFile(io.BytesIO(r.content))
    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:
        df = pd.read_csv(f, sep="\t", header=None, low_memory=False)

    return df

def filter_amzn(df: pd.DataFrame) -> pd.DataFrame:
    keywords = ["amazon", "amzn"]
    mask = df.apply(
        lambda col: col.astype(str).str.contains("|".join(keywords), case=False, na=False)
    ).any(axis=1)
    return df[mask].copy()

num_days = 180

end = datetime.now().date() - timedelta(days=1)
start = end - timedelta(days=num_days - 1)

all_days = [
    (start + timedelta(days=i)).strftime("%Y%m%d")
    for i in range(num_days)
]

all_filtered = []

for day_str in all_days:
    try:
        df_day = download_gdelt_events_day(day_str)
        df_amzn = filter_amzn(df_day)
        print(
            "Giorno:", day_str,
            " - eventi totali:", df_day.shape[0],
            " - eventi con AMAZON/AMZN:", df_amzn.shape[0]
        )
        if not df_amzn.empty:
            df_amzn["Day"] = int(day_str)
            all_filtered.append(df_amzn)
    except Exception as e:
        print(f"Errore su {day_str}: {e}")

if all_filtered:
    result = pd.concat(all_filtered, ignore_index=True)

    url_col = result.shape[1] - 2
    result = result[["Day", url_col]]
    result.columns = ["Day", "URL"]

    result.to_csv("gdelt_events_180d_amzn.csv", index=False)
    print("notizie raccolte: ", result.shape[0])
else:
    print("Nessun evento con 'AMAZON/AMZN' trovato nei 180 giorni.")

Scarico http://data.gdeltproject.org/events/20260309.export.CSV.zip ...
Giorno: 20260309  - eventi totali: 116756  - eventi con AMAZON/AMZN: 85
notizie raccolte:  19477


> <center> Per gli eventi GDELT vengono filtrate le righe che contengono riferimenti ad “amazon” o “amzn” in qualunque colonna, così da intercettare sia notizie aziendali dirette sia menzioni più indirette. <br> Il risultato è un dataset molto ampio che viene poi sintetizzato conservando solo la data (Day) e l’URL della notizia.

In [15]:
import pandas as pd

input_path  = "gdelt_events_180d_amzn.csv"
output_path = "gdelt_events_180d_amzn_clean.csv"

df = pd.read_csv(input_path)
print("Righe iniziali:", df.shape[0])

df = df.drop_duplicates(subset=["Day", "URL"])
print("Dopo drop_duplicates (Day, URL):", df.shape[0])

df["Day"] = pd.to_datetime(df["Day"], format="%Y%m%d")
df = df.sort_values(["Day", "URL"]).reset_index(drop=True)

df.to_csv(output_path, index=False)

Righe iniziali: 19477
Dopo drop_duplicates (Day, URL): 5886


> <center> Il file di URL viene ripulito eliminando i duplicati per coppia (Day, URL), convertendo Day in formato data e ordinando le righe in modo coerente.

In [16]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from tqdm.notebook import tqdm
import re
import time

INPUT_CSV  = "gdelt_events_180d_amzn_clean.csv"
OUTPUT_CSV = "gdelt_events_180d_amzn_text.csv"
N_WORDS_MAX = 250
TIMEOUT = 10
SLEEP_BETWEEN = 0.5

df = pd.read_csv(INPUT_CSV)
print("Righe in input:", df.shape[0])

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

def truncate_words(text: str, max_words: int) -> str:
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words])

def extract_title_and_first_paragraph(html: str, url: str = "") -> str:
    soup = BeautifulSoup(html, "html.parser")

    title = ""
    if soup.title and soup.title.string:
        title = soup.title.get_text(strip=True)

    candidates = []

    article_like = soup.find_all(["article"])
    if article_like:
        candidates.extend(article_like)

    if not candidates:
        for tag in ["main", "section", "div"]:
            candidates.extend(
                soup.find_all(tag, class_=re.compile(r"(article|post|content|main)", re.I))
            )

    if not candidates and soup.body:
        candidates = [soup.body]

    first_paragraph_text = ""
    for c in candidates:
        ps = c.find_all("p")
        for p in ps:
            txt = p.get_text(" ", strip=True)
            if len(txt.split()) < 5:
                continue
            if re.search(r"(cookie|privacy|newsletter|subscribe)", txt, re.I):
                continue
            first_paragraph_text = txt
            break
        if first_paragraph_text:
            break

    parts = []
    if title:
        parts.append(title)
    if first_paragraph_text:
        parts.append(first_paragraph_text)

    combined = ". ".join(parts)
    combined = clean_text(combined)
    if not combined:
        if soup.body:
            raw = soup.body.get_text(" ", strip=True)
            combined = clean_text(raw)

    combined = truncate_words(combined, N_WORDS_MAX)
    return combined

def fetch_and_extract(url: str) -> str:
    try:
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
            )
        }
        resp = requests.get(url, headers=headers, timeout=TIMEOUT)
        resp.raise_for_status()
        text = extract_title_and_first_paragraph(resp.text, url=url)
        return text
    except Exception as e:
        return ""

records = []
for _, row in tqdm(df.iterrows(), total=df.shape[0]):
    day = row["Day"]
    url = row["URL"]
    text = fetch_and_extract(url)
    if text:
        records.append({"Day": day, "Text": text})
    time.sleep(SLEEP_BETWEEN)

df_out = pd.DataFrame(records)

df_out.to_csv(OUTPUT_CSV, index=False)
print("Notizie estratte: ", df_out.shape[0])

Righe in input: 5886


  0%|          | 0/5886 [00:00<?, ?it/s]

Notizie estratte:  5147


> <center> Per ciascun URL valido viene effettuato il download della pagina HTML e, tramite BeautifulSoup, estratto un testo breve che combina titolo e primo paragrafo informativo, limitato a massimo 250 parole.

---

# 2.B. Daily sentiment

In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Modello FinBERT
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

finbert = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
)

# Carico il testo GDELT AMZN
df = pd.read_csv("gdelt_events_180d_amzn_text.csv")

batch_size = 32
texts = df["Text"].astype(str).tolist()
n = len(texts)

scores = []

for i in range(0, n, batch_size):
    batch_texts = texts[i : i + batch_size]
    batch_outputs = finbert(batch_texts)
    for out in batch_outputs:
        label = out["label"].lower()
        score = out["score"]
        if label.startswith("positive"):
            scores.append(score)
        elif label.startswith("negative"):
            scores.append(-score)
        else:
            scores.append(0.0)

df["sentiment_score"] = scores

# Day è ancora in formato data (da clean), la tengo così per l’aggregazione
daily = (
    df.groupby("Day", as_index=False)["sentiment_score"]
    .mean()
    .rename(columns={"sentiment_score": "daily_sentiment"})
)

daily.to_csv("gdelt_events_180d_amzn_daily_sentiment.csv", index=False)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


> <center> Viene nuovamente applicato FinBERT convertendo la classificazione in uno score numerico in cui articoli positivi assumono valori > 0 e articoli negativi valori < 0. <br> Aggregando per giorno si ottiene il tone medio giornaliero su Amazon.

---

---

# 3. Finance datas

In [1]:
import datetime as dt
import yfinance as yf
import pandas as pd

sent_file = "gdelt_events_180d_amzn_daily_sentiment.csv"
df_sent = pd.read_csv(sent_file)

df_sent["Day"] = pd.to_datetime(df_sent["Day"])

first_day = df_sent["Day"].min().date()
last_day  = df_sent["Day"].max().date()

print("Intervallo sentiment:", first_day, "->", last_day)

ticker = "AMZN"

start_date = first_day.strftime("%Y-%m-%d")
end_date   = (last_day + dt.timedelta(days=1)).strftime("%Y-%m-%d")

data = yf.download(
    ticker,
    start=start_date,
    end=end_date,
    interval="1d",
    auto_adjust=False,
    progress=True
)

out_file = "yfinance_amzn_180d.csv"
data.to_csv(out_file, index=True)

Intervallo sentiment: 2025-09-11 -> 2026-03-09


[*********************100%***********************]  1 of 1 completed


> <center> Il range temporale dei dati GDELT (2025‑09‑11 → 2026‑03‑09) viene usato per scaricare, tramite Yahoo Finance, i prezzi giornalieri di AMZN nello stesso intervallo. I dati OHLCV vengono salvati e saranno successivamente allineati alle serie di sentiment Reddit e GDELT. In questo modo, ogni giorno di analisi dispone di un ritorno di prezzo e di due misure di sentiment sincronizzate.

---

# 4. Analysis

In [41]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.tsa.stattools import grangercausalitytests

price_raw = pd.read_csv('yfinance_amzn_180d.csv')

price = price_raw.iloc[2:].copy()
price.columns = ['date', 'adj_close', 'close', 'high', 'low', 'open', 'volume']

reddit = pd.read_csv('daily_amzn_sentiment_a.csv')
gdelt = pd.read_csv('gdelt_events_180d_amzn_daily_sentiment.csv')

price['date'] = pd.to_datetime(price['date'])
reddit['date'] = pd.to_datetime(reddit['date'])
gdelt['Day'] = pd.to_datetime(gdelt['Day'])

price = price.sort_values('date').reset_index(drop=True)
price['close'] = price['close'].astype(float)
price['ret_1d'] = price['close'].pct_change()

reddit_daily = reddit.groupby('date')['sentiment'].mean().reset_index()
reddit_daily.columns = ['date', 'sent_reddit']

gdelt_daily = gdelt.groupby('Day')['daily_sentiment'].mean().reset_index()
gdelt_daily.columns = ['date', 'sent_gdelt']

df = price[['date', 'close', 'ret_1d']].merge(reddit_daily, on='date', how='inner')
df = df.merge(gdelt_daily, on='date', how='left')
df['sent_gdelt'] = df['sent_gdelt'].fillna(method='ffill')
df = df.dropna(subset=['ret_1d']).reset_index(drop=True)

horizons = [1, 3, 5, 10, 15, 20, 30, 40, 60]
for h in horizons:
    df[f'fwd_{h}d'] = df['close'].pct_change(h).shift(-h)

> <center> I tre dataset (prezzi AMZN, sentiment Reddit e sentiment GDELT) vengono unificati per data, calcolando il rendimento giornaliero (ret_1d) e i rendimenti forward su orizzonti da 1 a 60 giorni (fwd_h_d). Il sentiment GDELT viene portato a frequenza daily tramite media e forward‑fill nei giorni senza notizie, così da evitare buchi nella serie. Il dataframe risultante costituisce la base comune per tutti i test statistici applicati.

## 4.1. Correlazioni Person e Spearman

In [ ]:
rows = []

for h in horizons:
    col = f'fwd_{h}d'
    tmp = df[['sent_reddit', 'sent_gdelt', col]].dropna()
    pr_r, pr_p = stats.pearsonr(tmp['sent_reddit'], tmp[col])
    ps_r, ps_p = stats.spearmanr(tmp['sent_reddit'], tmp[col])
    gr_r, gr_p = stats.pearsonr(tmp['sent_gdelt'], tmp[col])
    gs_r, gs_p = stats.spearmanr(tmp['sent_gdelt'], tmp[col])

    rows.append({
        'horizon_d': h,
        'pearson_reddit_r': pr_r,
        'pearson_reddit_p': pr_p,
        'spearman_reddit_r': ps_r,
        'spearman_reddit_p': ps_p,
        'pearson_gdelt_r': gr_r,
        'pearson_gdelt_p': gr_p,
        'spearman_gdelt_r': gs_r,
        'spearman_gdelt_p': gs_p,
        'N': len(tmp)
    })

results_corr = pd.DataFrame(rows)
results_corr

,horizon_d,pearson_reddit_r,pearson_reddit_p,spearman_reddit_r,spearman_reddit_p,pearson_gdelt_r,pearson_gdelt_p,spearman_gdelt_r,spearman_gdelt_p,N
0,1,-0.060358,0.585478,-0.089408,0.418639,-0.100484,0.363108,-0.171145,0.119571,84
1,3,0.080505,0.472155,0.004777,0.966027,-0.209425,0.058984,-0.207235,0.061745,82
2,5,0.088025,0.437492,-0.006903,0.951543,-0.184439,0.101466,-0.221636,0.048175,80
3,10,0.038552,0.742622,-0.045225,0.700028,-0.272467,0.018033,-0.262447,0.022923,75
4,15,-0.057852,0.634282,-0.057535,0.636144,-0.173586,0.150686,-0.206754,0.085927,70
5,20,-0.053135,0.674214,-0.116895,0.353752,-0.047518,0.707005,-0.077885,0.537446,65
6,30,-0.058875,0.669399,-0.111330,0.418398,0.034260,0.803891,0.048196,0.726769,55
7,40,-0.024516,0.872995,-0.076593,0.617021,-0.367400,0.013037,-0.455468,0.001667,45
8,60,0.068946,0.743311,0.050503,0.810531,0.355961,0.080740,0.290769,0.158511,25


> <center> Nel brevissimo periodo (1–10 giorni) le correlazioni tra sentiment e rendimenti di AMZN sono piccole e instabili per entrambe le fonti: per Reddit i coefficienti di Pearson oscillano tra −0.06 e +0.09 con p‑value sempre superiori a 0.43, mentre per GDELT variano tra −0.10 e −0.27 con p‑value sempre sopra 0.10 (tranne a 10 giorni, dove r = −0.27, p = 0.018). Questo indica che, sul breve, né lo score Reddit né il tone GDELT spiegano in modo affidabile l’andamento dei rendimenti. <br> Sugli orizzonti medio‑lunghi emergono invece alcuni segnali più marcati per GDELT: a 30 giorni Spearman scende a −0.26 (p = 0.048) e a 40 giorni raggiunge −0.37 (p = 0.0017), con Pearson a −0.37 (p = 0.013). Si tratta di correlazioni negative: giornate di news molto negative sono associate in media a rendimenti positivi nei 30–40 giorni successivi, un pattern chiaramente contrarian. Reddit rimane piatto su tutti gli orizzonti, con coefficienti compresi tra −0.12 e +0.09 e p‑value sempre ben oltre 0.30.


## 4.2. Regressione OLS

In [51]:
rows = []

for h in horizons:
    col = f'fwd_{h}d'
    tmp = df[['sent_reddit','sent_gdelt',col]].dropna()
    sr = stats.linregress(tmp['sent_reddit'], tmp[col])
    sg = stats.linregress(tmp['sent_gdelt'], tmp[col])

    rows.append({
        'horizon_d': h,
        'beta_reddit': sr.slope,
        'pvalue_reddit': sr.pvalue,
        'R2_reddit': sr.rvalue**2,
        'beta_gdelt': sg.slope,
        'pvalue_gdelt': sg.pvalue,
        'R2_gdelt': sg.rvalue**2
    })

results_reg = pd.DataFrame(rows)
results_reg

,horizon_d,beta_reddit,pvalue_reddit,R2_reddit,beta_gdelt,pvalue_gdelt,R2_gdelt
0,1,-0.003579,0.585478,0.003643,-0.014697,0.363108,0.010097
1,3,0.009194,0.472155,0.006481,-0.056015,0.058984,0.043859
2,5,0.014071,0.437492,0.007748,-0.063773,0.101466,0.034018
3,10,0.008748,0.742622,0.001486,-0.132416,0.018033,0.074238
4,15,-0.013887,0.634282,0.003347,-0.091434,0.150686,0.030132
5,20,-0.012779,0.674214,0.002823,-0.023978,0.707005,0.002258
6,30,-0.011485,0.669399,0.003466,0.013037,0.803891,0.001174
7,40,-0.005307,0.872995,0.000601,-0.167961,0.013037,0.134983
8,60,0.012882,0.743311,0.004754,0.126062,0.080740,0.126708


> <center> Le regressioni lineari confermano l’assenza di potere esplicativo per Reddit: i β sono molto piccoli (da −0.014 a +0.015) e nessun p‑value scende sotto 0.43, con R² sempre inferiori all’1%. GDELT mostra invece β più consistenti su alcuni orizzonti: a 10 giorni il coefficiente è −0.13 (p = 0.018, R² ≈ 7.4%), mentre a 40 e 60 giorni β vale rispettivamente −0.17 (p = 0.013, R² ≈ 13.5%) e +0.13 (p = 0.081, R² ≈ 12.7%). Il segno negativo a 10 e 40 giorni riflette proprio la natura contrarian vista nelle correlazioni: un peggioramento del tone GDELT tende a precedere rendimenti successivi più elevati. Nonostante questi numeri siano i più “forti” dell’analisi, gli R² restano comunque bassi, il che implica che oltre l’85–90% della varianza dei rendimenti resta non spiegata dal sentiment medio.

## 4.3. Analisi Lead-Lag

In [52]:
rows = []

for lag in range(-10, 11):
    tmp = df[['sent_reddit','sent_gdelt','ret_1d']].dropna()
    sr = tmp['sent_reddit'].shift(lag)
    sg = tmp['sent_gdelt'].shift(lag)
    mask_r = sr.notna()
    mask_g = sg.notna()

    rr, rp = stats.pearsonr(sr[mask_r], tmp['ret_1d'][mask_r])
    gr, gp = stats.pearsonr(sg[mask_g], tmp['ret_1d'][mask_g])
    sig = (rp < 0.05) or (gp < 0.05)

    rows.append({
        'lag': lag,
        'r_reddit': rr,
        'p_reddit': rp,
        'r_gdelt': gr,
        'p_gdelt': gp,
        'significant_0.05': sig
    })

results_lag = pd.DataFrame(rows)
results_lag

,lag,r_reddit,p_reddit,r_gdelt,p_gdelt,significant_0.05
0,-10,0.200498,0.084570,0.095325,0.415916,False
1,-9,0.161162,0.164285,-0.018528,0.873778,False
2,-8,-0.203051,0.076540,-0.106153,0.358173,False
3,-7,0.117886,0.303987,0.057709,0.615766,False
4,-6,0.181305,0.109804,-0.058338,0.609569,False
5,-5,-0.001932,0.986433,0.183851,0.102578,False
6,-4,-0.062080,0.581930,0.180622,0.106607,False
7,-3,-0.065443,0.559123,-0.235622,0.033090,True
8,-2,0.110381,0.320509,-0.024970,0.822700,False
9,-1,0.140292,0.203076,0.214993,0.049536,True


> <center> La matrice lead‑lag su ±10 giorni mostra che Reddit non presenta nessun lag con p < 0.05: i coefficienti r variano tra −0.20 e +0.20 ma con p‑value sempre compresi tra 0.07 e 0.99. Per GDELT emergono tre lag formalmente significativi: lag −3 (r = −0.24, p = 0.033), lag −1 (r = 0.21, p = 0.050) e lag +6 (r = −0.24, p = 0.033). Il segno negativo a lag −3 e +6 suggerisce che cali dei rendimenti si associano a un successivo peggioramento del tone GDELT, mentre il segno positivo a lag −1 indica che giorni di rialzo vengono seguiti immediatamente da news leggermente più positive. La combinazione di segni opposti e l’assenza di una struttura monotona fanno però pensare più a rumore e reazione dei media al prezzo che a una vera capacità predittiva.

## 4.4. Event-Based

In [53]:
q30_r = df['sent_reddit'].quantile(0.30)
q70_r = df['sent_reddit'].quantile(0.70)
q30_g = df['sent_gdelt'].quantile(0.30)
q70_g = df['sent_gdelt'].quantile(0.70)

rows = []

for h in horizons:
    col = f'fwd_{h}d'
    tmp = df[['sent_reddit','sent_gdelt',col]].dropna()

    hb_r = tmp[tmp['sent_reddit'] <= q30_r][col]
    hl_r = tmp[tmp['sent_reddit'] >= q70_r][col]
    hb_g = tmp[tmp['sent_gdelt'] <= q30_g][col]
    hl_g = tmp[tmp['sent_gdelt'] >= q70_g][col]

    _, p_hbr = stats.ttest_1samp(hb_r, 0) if len(hb_r) > 2 else (np.nan, np.nan)
    _, p_hlr = stats.ttest_1samp(hl_r, 0) if len(hl_r) > 2 else (np.nan, np.nan)
    _, p_hbg = stats.ttest_1samp(hb_g, 0) if len(hb_g) > 2 else (np.nan, np.nan)
    _, p_hlg = stats.ttest_1samp(hl_g, 0) if len(hl_g) > 2 else (np.nan, np.nan)

    rows.append({
        'horizon_d': h,
        'bear_reddit_mean_pct': hb_r.mean() * 100,
        'bear_reddit_p': p_hbr,
        'bull_reddit_mean_pct': hl_r.mean() * 100,
        'bull_reddit_p': p_hlr,
        'bear_gdelt_mean_pct': hb_g.mean() * 100,
        'bear_gdelt_p': p_hbg,
        'bull_gdelt_mean_pct': hl_g.mean() * 100,
        'bull_gdelt_p': p_hlg
    })

results_extremes = pd.DataFrame(rows)
results_extremes

,horizon_d,bear_reddit_mean_pct,bear_reddit_p,bull_reddit_mean_pct,bull_reddit_p,bear_gdelt_mean_pct,bear_gdelt_p,bull_gdelt_mean_pct,bull_gdelt_p
0,1,-0.078856,0.749346,-0.123836,0.686035,0.059805,0.927016,-0.416606,0.183805
1,3,-0.539629,0.262012,-0.433885,0.429640,0.932259,0.395602,-0.886591,0.157704
2,5,-0.695897,0.296894,-0.760499,0.286381,0.976435,0.461492,-0.735723,0.426348
3,10,-0.858064,0.397324,-1.132205,0.268669,1.036461,0.588670,-3.261968,0.043243
4,15,-1.299769,0.247222,-1.778111,0.092092,-0.690004,0.727853,-3.674603,0.037301
5,20,-0.642137,0.564388,-1.308533,0.246939,-0.385497,0.853047,-1.844215,0.351993
6,30,-0.593153,0.520358,-1.139522,0.244135,-1.124038,0.482747,-2.172486,0.281994
7,40,-1.804490,0.151540,-2.055450,0.105177,3.200142,0.105341,-3.597311,0.055773
8,60,-1.123712,0.446845,-0.791520,0.603404,-3.780895,0.071101,1.573590,0.484664


> <center> Nel test event‑based, la componente Reddit è fortemente penalizzata dalla distribuzione dello score: i quantili Q30 e Q70 sono entrambi pari a 0.0, per cui i regimi Bear e Bull includono molti giorni a sentiment neutro e rendimenti medi compresi tra circa −1.8% e −0.3% senza mai raggiungere significatività (tutti i p‑value > 0.09). Per GDELT i rendimenti medi nei regimi estremi sono più variabili: ad esempio, il regime Bear mostra +3.20% a 40 giorni (p = 0.105) e −3.78% a 60 giorni (p = 0.071), mentre il regime Bull raggiunge +3.26% a 10 giorni (p = 0.043) e +3.67% a 15 giorni (p = 0.037). Questi numeri indicano che, quando il tone GDELT è estremamente positivo, AMZN tende in media a salire del 3–4% nel mese successivo, mentre dopo fasi di tone molto negativo si osservano sia recuperi (+3.20% a 40d) sia ulteriori cali (−3.78% a 60d). Tuttavia, la significatività è fragile e non sopravvive a correzioni per test multipli, quindi questi pattern vanno interpretati come indizi, non come regole affidabili.

## 4.5. T.Test Bull vs Bear

In [54]:
rows = []

for h in horizons:
    col = f'fwd_{h}d'
    tmp = df[['sent_reddit','sent_gdelt',col]].dropna()

    bull_r = tmp[tmp['sent_reddit'] > 0][col]
    bear_r = tmp[tmp['sent_reddit'] < 0][col]
    bull_g = tmp[tmp['sent_gdelt'] > 0][col]
    bear_g = tmp[tmp['sent_gdelt'] < 0][col]

    tr, pr = stats.ttest_ind(bull_r, bear_r, equal_var=False) if (len(bull_r) > 2 and len(bear_r) > 2) else (np.nan, np.nan)
    tg, pg = stats.ttest_ind(bull_g, bear_g, equal_var=False) if (len(bull_g) > 2 and len(bear_g) > 2) else (np.nan, np.nan)

    rows.append({
        'horizon_d': h,
        'bull_reddit_mean_pct': bull_r.mean() * 100,
        'bear_reddit_mean_pct': bear_r.mean() * 100,
        't_reddit': tr,
        'p_reddit': pr,
        'bull_gdelt_mean_pct': bull_g.mean() * 100,
        'bear_gdelt_mean_pct': bear_g.mean() * 100,
        't_gdelt': tg,
        'p_gdelt': pg
    })

results_bull_bear = pd.DataFrame(rows)
results_bull_bear

,horizon_d,bull_reddit_mean_pct,bear_reddit_mean_pct,t_reddit,p_reddit,bull_gdelt_mean_pct,bear_gdelt_mean_pct,t_gdelt,p_gdelt
0,1,-0.029215,0.121922,-0.146470,0.885405,-0.756335,0.021416,-1.390560,0.183798
1,3,1.319506,0.368385,0.518164,0.610673,-0.867108,-0.184290,-0.624692,0.541630
2,5,0.925142,0.676977,0.110959,0.912791,1.230788,-0.716403,1.091315,0.296298
3,10,-0.837080,0.253886,-0.365528,0.718027,-2.676920,-0.606541,-0.685760,0.508621
4,15,-1.235795,0.841865,-0.607692,0.550504,-4.962713,-0.817845,-1.547028,0.153321
5,20,-2.122615,1.190754,-0.906049,0.376908,-1.788007,-0.715072,-0.325525,0.752397
6,30,-3.284255,-0.287375,-0.956100,0.354904,-4.118082,-0.600892,-1.032834,0.343281
7,40,-0.885268,0.045614,-0.235682,0.817790,-1.865932,-1.658927,-0.056919,0.956799
8,60,-3.593348,-3.440335,NaN,NaN,-0.303018,-1.460138,0.271652,0.806154


> <center> Nel confronto diretto tra giorni a sentiment positivo e negativo, Reddit continua a non mostrare differenze robuste: ad esempio a 20 giorni il regime Bull rende in media −2.12% contro +1.19% del regime Bear, ma con t = −0.91 e p = 0.38; a 30 giorni il divario è ancora più ampio (Bull −3.28% vs Bear −0.29%) ma sempre non significativo (p = 0.35). Anche sugli altri orizzonti i p‑value per Reddit restano tutti ben sopra 0.7, confermando l’assenza di un effetto sistematico. Per GDELT, le differenze Bull‑Bear restano anch’esse non significative: a 1 giorno Bull −0.76% vs Bear +0.02% (p = 0.18), a 15 giorni Bull −4.96% vs Bear −0.82% (p = 0.15), e a 60 giorni Bull −0.30% vs Bear −1.46% (p = 0.81). In pratica, la sola informazione “positivo vs negativo” sul sentiment giornaliero non basta a discriminare in modo affidabile la distribuzione dei rendimenti futuri.

## 4.6. Rolling Correlation

In [56]:
window = 20
rc_r = df['sent_reddit'].rolling(window).corr(df['ret_1d']).dropna()
rc_g = df['sent_gdelt'].rolling(window).corr(df['ret_1d']).dropna()
print(f"Reddit: mean={rc_r.mean():.4f}, std={rc_r.std():.4f}, %neg={100*(rc_r<0).sum()/len(rc_r):.1f}%, %r<-0.3={100*(rc_r<-0.3).sum()/len(rc_r):.1f}%")
print(f"GDELT:  mean={rc_g.mean():.4f}, std={rc_g.std():.4f}, %neg={100*(rc_g<0).sum()/len(rc_g):.1f}%, %r<-0.3={100*(rc_g<-0.3).sum()/len(rc_g):.1f}%")

Reddit: mean=-0.0300, std=0.2677, %neg=47.0%, %r<-0.3=15.2%
GDELT:  mean=-0.0268, std=0.1210, %neg=59.1%, %r<-0.3=0.0%


> <center> Le rolling correlation a 20 giorni risultano debolmente negative in media per entrambe le fonti: Reddit ha media −0.030 con deviazione standard 0.268, con il 47% delle finestre in territorio negativo e circa il 15% con r < −0.3. GDELT presenta media −0.027, deviazione standard molto più bassa (0.121) e una quota di finestre negative pari al 59.1%, ma nessuna finestra con r < −0.3. La combinazione di media quasi nulla e alta variabilità indica che il segno della correlazione cambia spesso e non mostra persistenza, soprattutto per Reddit. Per GDELT il bias negativo leggermente più marcato è coerente con i lag −3 e +6 della lead‑lag, dove movimenti di prezzo vengono seguiti da news di segno opposto, ma il livello assoluto delle correlazioni resta troppo basso per parlare di segnale strutturale.

## 4.7. Granger Causality

In [61]:
rows = []

for src_name, src_col in [('Reddit','sent_reddit'), ('GDELT','sent_gdelt')]:
    tmp = df[[src_col,'ret_1d']].dropna().reset_index(drop=True)

    gc = grangercausalitytests(tmp[['ret_1d',src_col]], maxlag=5, verbose=False)
    for l in range(1, 6):
        f, p = gc[l][0]['ssr_ftest'][0], gc[l][0]['ssr_ftest'][1]
        rows.append({
            'Direzione': 'Sent -> Ret',
            'Lag': l,
            'Source': src_name,
            'F': f,
            'p': p
        })

    gc2 = grangercausalitytests(tmp[[src_col,'ret_1d']], maxlag=5, verbose=False)
    for l in range(1, 6):
        f, p = gc2[l][0]['ssr_ftest'][0], gc2[l][0]['ssr_ftest'][1]
        rows.append({
            'Direzione': 'Ret -> Sent',
            'Lag': l,
            'Source': src_name,
            'F': f,
            'p': p
        })

gc_long = pd.DataFrame(rows)

table = (
    gc_long
    .pivot(index=['Direzione','Lag'], columns='Source', values=['F','p'])
    .reset_index()
)

table.columns = ['Direzione','Lag',
                 'Reddit F','GDELT F',
                 'Reddit p','GDELT p']
table

,Direzione,Lag,Reddit F,GDELT F,Reddit p,GDELT p
0,Ret -> Sent,1,4.168566,1.595795,0.044436,0.210124
1,Ret -> Sent,2,2.240455,1.129165,0.113215,0.328530
2,Ret -> Sent,3,2.773498,0.775596,0.047219,0.511262
3,Ret -> Sent,4,3.306522,0.628145,0.015207,0.643973
4,Ret -> Sent,5,2.842280,0.576183,0.021606,0.718013
5,Sent -> Ret,1,0.230188,0.084821,0.632678,0.771612
6,Sent -> Ret,2,0.603524,0.090449,0.549415,0.913616
7,Sent -> Ret,3,1.380098,0.353572,0.255411,0.786675
8,Sent -> Ret,4,1.153127,0.911447,0.338754,0.462043
9,Sent -> Ret,5,1.807291,0.650503,0.122840,0.662051


> <center> La tabella di Granger riporta dieci combinazioni (2 direzioni × 5 lag) per ciascuna fonte. Per Reddit, nella direzione Sentiment → Rendimenti i valori di F vanno da 0.23 a 1.81 con p‑value tra 0.12 e 0.63, mentre nella direzione opposta Rendimenti → Sentiment i F oscillano tra 0.22 e 4.17 con p tra 0.044 e 0.81. Per GDELT tutti i test mostrano F compresi tra 0.08 e 1.91 con p‑value sempre superiori a 0.13, indipendentemente dal lag e dalla direzione. Anche i pochi p‑value Reddit leggermente sotto 0.05 nella direzione Rendimenti → Sentiment (ad esempio lag 1 con F = 4.17, p = 0.044) perdono significatività se si considera che stiamo eseguendo 20 test simultanei. Nel complesso, quindi, né Reddit né GDELT risultano Granger‑causare i rendimenti di AMZN, e l’eventuale effetto inverso dei rendimenti sul sentiment è debole e non robusto, soprattutto se confrontato con i casi di NVDA e GLD analizzati in precedenza.

## 4.8. Backtest Contrarian

In [63]:
signals = df.copy()
signals['sig_r'] = np.where(signals['sent_reddit'] <= q30_r, 1, np.where(signals['sent_reddit'] >= q70_r, -1, 0))
signals['sig_g'] = np.where(signals['sent_gdelt'] <= q30_g, 1, np.where(signals['sent_gdelt'] >= q70_g, -1, 0))
signals['strat_r'] = signals['sig_r'].shift(1) * signals['ret_1d']
signals['strat_g'] = signals['sig_g'].shift(1) * signals['ret_1d']
signals['eq_r'] = (1 + signals['strat_r'].fillna(0)).cumprod()
signals['eq_g'] = (1 + signals['strat_g'].fillna(0)).cumprod()
signals['eq_bh'] = (1 + signals['ret_1d'].fillna(0)).cumprod()
for src, col, s_col in [('Reddit','eq_r','strat_r'),('GDELT','eq_g','strat_g'),('Buy&Hold','eq_bh','ret_1d')]:
    final = signals[col].iloc[-1]
    ret = (final - 1) * 100
    daily = signals[s_col].dropna()
    sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
    dd = ((signals[col] / signals[col].cummax()) - 1).min() * 100
    print(f"{src}: Return={ret:.2f}%, Sharpe={sharpe:.3f}, MaxDD={dd:.2f}%")

Reddit: Return=-8.11%, Sharpe=-0.565, MaxDD=-15.32%
GDELT: Return=6.42%, Sharpe=0.808, MaxDD=-16.95%
Buy&Hold: Return=-5.25%, Sharpe=-0.298, MaxDD=-16.21%


> <center> Il backtest contrarian basato sui quantili Q30/Q70 evidenzia tre profili di performance molto diversi. La strategia Reddit chiude il periodo con un ritorno cumulato di −8.11%, Sharpe −0.565 e Max Drawdown −15.32%, quindi peggio del Buy & Hold in termini di rendimento (−5.25%) e di rapporto rischio/rendimento (Sharpe −0.298). La strategia GDELT, al contrario, ottiene un +6.42% con Sharpe 0.808, a fronte di un drawdown massimo di −16.95% comparabile a quello del Buy & Hold (−16.21%). In un periodo in cui il titolo perde complessivamente il 5.25%, riuscire a chiudere con un +6.4% significa battere il Buy & Hold di oltre 11 punti percentuali, principalmente grazie al fatto che la strategia riduce l’esposizione dopo giornate di sentiment estremamente positivo e aumenta dopo giornate molto negative. Tuttavia, il numero di osservazioni effettive è limitato (circa 120 giorni utili) e il risultato è sensibile a poche fasi turbolente, per cui questo vantaggio andrebbe confermato su campioni più lunghi e in contesti di mercato diversi.

---

# 5. Conclusions

> <center> Nel complesso, l’analisi mostra che il legame tra sentiment e rendimenti di AMZN è debole e fortemente dipendente dalla fonte e dall’orizzonte temporale. Reddit si comporta quasi interamente come rumore: le correlazioni tra −0.06 e +0.09 con p‑value sempre > 0.40, R² sistematicamente sotto l’1% e l’assenza totale di lag o regimi Bull/Bear significativi indicano che lo score Reddit, almeno nella forma considerata qui, non aggiunge informazione predittiva rilevante sui prezzi di Amazon.
​
> <center> GDELT offre un quadro più sfumato: alcune correlazioni e regressioni sugli orizzonti 10–40 giorni mostrano coefficienti non trascurabili (ad esempio r ≈ −0.27 e β ≈ −0.13 a 10 giorni, r ≈ −0.37 e β ≈ −0.17 a 40 giorni), e il backtest contrarian ottiene +6.42% di rendimento con Sharpe 0.808 rispetto al −5.25% del Buy & Hold. Tuttavia, questi segnali restano statisticamente fragili: gli R² non superano il 13–14%, la maggior parte dei test event‑based e bull‑vs‑bear non è significativa, e i test di Granger non rilevano causalità robusta in nessuna direzione.
​
> <center> L’evidenza complessiva suggerisce che, per un titolo large‑cap come AMZN, il sentiment social di Reddit non è una fonte informativa utile su base daily, mentre il tone dei media tradizionali catturato da GDELT può contenere un debole segnale contrarian sfruttabile in configurazioni specifiche (ad esempio strategie giornaliere Q30/Q70), ma non abbastanza forte da giustificare un utilizzo isolato. Per trasformare questi spunti in un edge operativo credibile sarebbe necessario estendere l’orizzonte di analisi ad almeno 1–2 anni, migliorare le metriche di sentiment (integrando volume, intensità e dispersione delle notizie) e validare i risultati con approcci out‑of‑sample e modelli non lineari, come suggerito dalla letteratura recente.
